# Syntecxhub Internship — Website Traffic Analysis

**Project:** Website Traffic Analysis  
**Internship:** Syntecxhub Data Analytics Internship

## Objective
Analyze website traffic and user engagement to identify traffic trends, high-performing channels, engagement patterns, peak traffic hours, and the relationship between traffic volume and engagement.

This notebook answers the seven analytical questions defined in the provided website-performance reference and uses the supplied dataset rather than invented results.


## 1. Import Libraries

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid", context="notebook")

# Resolve the project from the notebook working directory or one of its parents.
project_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "csv").is_dir() and (path / "Website_Traffic_Analysis.ipynb").is_file()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "Could not locate the project root (expected Website_Traffic_Analysis.ipynb and csv/ "
        f"from the current directory or its parents: {Path.cwd()})"
    )

csv_files = sorted((project_root / "csv").glob("*.csv"))
if len(csv_files) != 1:
    raise FileNotFoundError(
        f"Expected one source CSV in {project_root / 'csv'}, found {len(csv_files)}: {csv_files}"
    )

DATA_PATH = csv_files[0]
VISUALIZATION_DIR = project_root / "visualizations"
DASHBOARD_DIR = project_root / "dashboard"
VISUALIZATION_DIR.mkdir(parents=True, exist_ok=True)
DASHBOARD_DIR.mkdir(parents=True, exist_ok=True)


def save_chart(filename):
    """Save the current Matplotlib figure in the project chart folder."""
    output_path = VISUALIZATION_DIR / filename
    plt.savefig(output_path, dpi=180, bbox_inches="tight")
    return output_path


print(f"Project folder: {project_root}")
print(f"Source dataset: {DATA_PATH.relative_to(project_root)}")

## 2. Load and Inspect the Dataset

The supplied CSV contains one descriptive/comment row followed by the actual column-header row. We therefore read it without assuming the first line is the header and explicitly select the real header row.


In [ ]:
# Read all rows as text first so the descriptive row and true header can be inspected.
raw = pd.read_csv(DATA_PATH, header=None, dtype=str, keep_default_na=False)
print("Raw CSV shape (including description and header):", raw.shape)
display(raw.head(4))

In [ ]:
# Locate the real header by its field label instead of assuming a fixed column count.
header_candidates = raw.index[
    raw.iloc[:, 0].str.casefold().str.contains("channel group", na=False)
]
if len(header_candidates) != 1:
    raise ValueError(
        f"Expected one channel-group header row in {DATA_PATH.name}; found {len(header_candidates)}."
    )

header_row = int(header_candidates[0])
source_headers = raw.iloc[header_row].astype(str).str.strip().tolist()
df = raw.iloc[header_row + 1 :].copy()
df.columns = source_headers
df = df.replace(r"^\s*$", np.nan, regex=True).dropna(how="all").reset_index(drop=True)

print(f"Detected header row: {header_row + 1}")
print(f"Observed source columns ({len(source_headers)}): {source_headers}")
display(df.head())

## 3. Rename Columns

The long Google Analytics export names are renamed to the terminology used in the analysis.


In [ ]:
# Normalize the known source fields while retaining any additional dataset columns.
source_to_canonical = {
    "users": "Users",
    "sessions": "Sessions",
    "engaged sessions": "Engaged Sessions",
    "average engagement time per session": "Average engagement time per session",
    "engaged sessions per user": "Engaged sessions per user",
    "events per session": "Events per session",
    "engagement rate": "Engagement rate",
    "event count": "Event count",
}
rename_map = {}
for source_column in df.columns:
    normalized = source_column.strip().casefold()
    if "channel group" in normalized:
        rename_map[source_column] = "channel group"
    elif normalized.startswith("date") and "hour" in normalized:
        rename_map[source_column] = "DateHour"
    elif normalized in source_to_canonical:
        rename_map[source_column] = source_to_canonical[normalized]

df = df.rename(columns=rename_map)
required_columns = [
    "channel group",
    "DateHour",
    "Users",
    "Sessions",
    "Engaged Sessions",
    "Average engagement time per session",
    "Engaged sessions per user",
    "Events per session",
    "Engagement rate",
    "Event count",
]
missing_required = [column for column in required_columns if column not in df.columns]
if missing_required:
    raise ValueError(f"Required analysis fields are missing from the CSV: {missing_required}")

print("Analysis fields found:", required_columns)
print("Additional source fields preserved:", [c for c in df.columns if c not in required_columns])
print("Rows and columns before cleaning:", df.shape)
print("Data types before conversion:")
print(df.dtypes.to_string())

## 4. Data Cleaning and Validation

In [ ]:
# Parse dates and convert available numeric fields without replacing source values silently.
df["channel group"] = df["channel group"].astype("string").str.strip()
df["DateHour"] = pd.to_datetime(
    df["DateHour"].astype("string").str.strip(),
    format="%Y%m%d%H",
    errors="coerce",
)
numeric_cols = [
    "Users",
    "Sessions",
    "Engaged Sessions",
    "Average engagement time per session",
    "Engaged sessions per user",
    "Events per session",
    "Engagement rate",
    "Event count",
]
for column in numeric_cols:
    df[column] = pd.to_numeric(df[column], errors="coerce")

required_for_analysis = ["channel group", "DateHour", "Users", "Sessions", "Engaged Sessions"]
invalid_required = df[required_for_analysis].isna().any(axis=1) | df["channel group"].eq("")
invalid_counts = df[["Users", "Sessions", "Engaged Sessions", "Event count"]].lt(0).any(axis=1)
invalid_logic = df["Engaged Sessions"].gt(df["Sessions"])
invalid_rate = df["Engagement rate"].notna() & ~df["Engagement rate"].between(0, 1)
invalid_rows = invalid_required | invalid_counts | invalid_logic | invalid_rate
print("Rows with invalid required values or impossible metrics:", int(invalid_rows.sum()))
df = df.loc[~invalid_rows].drop_duplicates().reset_index(drop=True)

# Date and Hour are derived from the validated timestamp for temporal analysis.
df["Date"] = df["DateHour"].dt.date
df["Hour"] = df["DateHour"].dt.hour
df["Non-Engaged Sessions"] = df["Sessions"] - df["Engaged Sessions"]

assert not df["DateHour"].isna().any()
assert (df["Engaged Sessions"] <= df["Sessions"]).all()
assert (df["Non-Engaged Sessions"] >= 0).all()

print("Cleaned shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Data types:")
print(df.dtypes.to_string())
print("Missing values by column:")
display(df.isna().sum().to_frame("Missing values"))
print("Exact duplicate rows remaining:", int(df.duplicated().sum()))
print("Summary statistics:")
display(df.describe(include="all").T)

## 5. Descriptive Statistics

In [ ]:
display(df.describe(include="all").T)


## 6. KPI Summary

In [ ]:
total_users = df["Users"].sum()
total_sessions = df["Sessions"].sum()
total_engaged = df["Engaged Sessions"].sum()
overall_engagement_rate = total_engaged / total_sessions
valid_time_rows = df["Average engagement time per session"].notna() & df["Sessions"].gt(0)
avg_engagement_time = np.average(
    df.loc[valid_time_rows, "Average engagement time per session"],
    weights=df.loc[valid_time_rows, "Sessions"],
)

kpis = pd.DataFrame(
    {
        "Metric": [
            "Total Users (sum of channel-hour counts)",
            "Total Sessions",
            "Total Engaged Sessions",
            "Overall Engagement Rate",
            "Weighted Average Engagement Time (seconds)",
            "Number of Traffic Channels",
            "Date Range",
        ],
        "Value": [
            int(total_users),
            int(total_sessions),
            int(total_engaged),
            overall_engagement_rate,
            avg_engagement_time,
            int(df["channel group"].nunique()),
            f"{df['DateHour'].min():%Y-%m-%d} to {df['DateHour'].max():%Y-%m-%d}",
        ],
    }
)
display(kpis)
print("Note: summed Users are not deduplicated across channel-hour records.")

# Question 1 — What patterns or trends can you observe in website sessions and users over time?

In [ ]:
trend = (
    df.groupby("DateHour", as_index=False)[["Sessions", "Users"]]
    .sum()
    .sort_values("DateHour")
)

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(trend["DateHour"], trend["Sessions"], label="Sessions", color="#1769aa")
ax.plot(trend["DateHour"], trend["Users"], label="Users (summed records)", color="#ef6c00")
ax.set_title("Users and Sessions Over Time", weight="bold")
ax.set_xlabel("Date and hour")
ax.set_ylabel("Count across channel-hour records")
ax.legend()
fig.autofmt_xdate()
fig.tight_layout()
save_chart("users_sessions_over_time.png")
plt.show()

print(
    "Interpretation: traffic varies by hour; users and sessions should be read as "
    "summed channel-hour counts, not unique people across the full date range."
)

In [ ]:
print("Peak sessions timestamp:")
display(trend.loc[trend["Sessions"].idxmax()])

print("Peak users timestamp:")
display(trend.loc[trend["Users"].idxmax()])


# Question 2 — Which marketing channel brought the highest number of users?

In [ ]:
channel_users = df.groupby("channel group")["Users"].sum().sort_values(ascending=False)
channel_sessions = df.groupby("channel group")["Sessions"].sum().sort_values(ascending=False)

display(channel_users.to_frame("Users (summed channel-hour counts)"))

fig, ax = plt.subplots(figsize=(9, 5))
channel_users.sort_values().plot(kind="barh", ax=ax, color="#1769aa")
ax.set_title("Users by Traffic Channel", weight="bold")
ax.set_xlabel("Users (summed channel-hour counts)")
ax.set_ylabel("Channel")
fig.tight_layout()
save_chart("users_by_channel.png")
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
channel_sessions.sort_values().plot(kind="barh", ax=ax, color="#2e8b57")
ax.set_title("Sessions by Traffic Channel", weight="bold")
ax.set_xlabel("Sessions")
ax.set_ylabel("Channel")
fig.tight_layout()
save_chart("sessions_by_channel.png")
plt.show()

In [ ]:
top_user_channel = channel_users.idxmax()
top_user_value = channel_users.max()

print(f"Top channel by users: {top_user_channel} ({top_user_value:,.0f} users)")


# Question 3 — Which channel has the highest average engagement time?

In [ ]:
weighted_time = df.assign(
    _weighted_time=df["Average engagement time per session"] * df["Sessions"]
).groupby("channel group").agg(
    weighted_seconds=("_weighted_time", "sum"),
    weighted_sessions=("Sessions", lambda values: values[df.loc[values.index, "Average engagement time per session"].notna()].sum()),
)
avg_time_by_channel = (
    weighted_time["weighted_seconds"] / weighted_time["weighted_sessions"]
).sort_values(ascending=False)

display(avg_time_by_channel.to_frame("Weighted Average Engagement Time (seconds)"))
fig, ax = plt.subplots(figsize=(9, 5))
avg_time_by_channel.sort_values().plot(kind="barh", ax=ax, color="#7653a6")
ax.set_title("Average Engagement Time by Channel", weight="bold")
ax.set_xlabel("Seconds per session (session-weighted)")
ax.set_ylabel("Channel")
fig.tight_layout()
save_chart("avg_engagement_time_by_channel.png")
plt.show()
print(
    f"Highest weighted average engagement time: {avg_time_by_channel.idxmax()} "
    f"({avg_time_by_channel.max():.1f} seconds). Interpret this with the channel's session volume."
)

# Question 4 — How does engagement rate vary across different traffic channels?

In [ ]:
channel_engagement = df.groupby("channel group").agg(
    Sessions=("Sessions", "sum"),
    Engaged_Sessions=("Engaged Sessions", "sum"),
)
channel_engagement["Engagement Rate"] = (
    channel_engagement["Engaged_Sessions"] / channel_engagement["Sessions"]
)
engagement_by_channel = channel_engagement["Engagement Rate"].sort_values(ascending=False)

display((engagement_by_channel * 100).round(2).to_frame("Engagement Rate (%)"))
fig, ax = plt.subplots(figsize=(9, 5))
(engagement_by_channel * 100).sort_values().plot(kind="barh", ax=ax, color="#008c95")
ax.set_title("Engagement Rate by Channel", weight="bold")
ax.set_xlabel("Engaged sessions / sessions (%)")
ax.set_ylabel("Channel")
fig.tight_layout()
save_chart("engagement_rate_by_channel.png")
plt.show()
print(
    "Rates are aggregated as total engaged sessions divided by total sessions; "
    "small-volume channels require extra caution."
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=df, x="channel group", y="Engagement rate", ax=ax, color="#b7d6e8")
ax.set_title("Hourly Engagement Rate Distribution by Channel", weight="bold")
ax.set_xlabel("Channel")
ax.set_ylabel("Engagement rate")
ax.tick_params(axis="x", rotation=35)
fig.tight_layout()
save_chart("engagement_rate_distribution_by_channel.png")
plt.show()

# Question 5 — Which channels drive more engaged sessions compared with non-engaged sessions?

In [ ]:
session_by_channel = df.groupby("channel group").agg(
    Sessions=("Sessions", "sum"),
    **{"Engaged Sessions": ("Engaged Sessions", "sum")},
)
session_by_channel["Non-Engaged Sessions"] = (
    session_by_channel["Sessions"] - session_by_channel["Engaged Sessions"]
)
session_by_channel["Engagement Rate"] = (
    session_by_channel["Engaged Sessions"] / session_by_channel["Sessions"]
)
display(session_by_channel.sort_values("Sessions", ascending=False))
print(
    f"Most engaged sessions by volume: {session_by_channel['Engaged Sessions'].idxmax()}; "
    f"lowest engagement rate: {session_by_channel['Engagement Rate'].idxmin()}."
)

In [ ]:
plot_df = session_by_channel[["Engaged Sessions", "Non-Engaged Sessions"]].reset_index()
plot_df = plot_df.melt(
    id_vars="channel group", var_name="Session Type", value_name="Sessions"
)
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=plot_df, x="channel group", y="Sessions", hue="Session Type", ax=ax)
ax.set_title("Engaged vs Non-Engaged Sessions by Channel", weight="bold")
ax.set_xlabel("Channel")
ax.set_ylabel("Sessions")
ax.tick_params(axis="x", rotation=35)
ax.legend(title="Session type")
fig.tight_layout()
save_chart("engaged_vs_non_engaged.png")
plt.show()
print(
    "Interpretation: compare both session counts and engagement rates; large channels "
    "naturally contribute more sessions, while rates support fairer quality comparisons."
)

# Question 6 — At what hours of the day does each channel drive the most traffic?

In [ ]:
hourly_channel = (
    df.groupby(["Hour", "channel group"])["Sessions"]
    .sum()
    .unstack(fill_value=0)
    .sort_index()
)
display(hourly_channel)
fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(hourly_channel, annot=True, fmt=",.0f", cmap="YlGnBu", linewidths=0.3, ax=ax)
ax.set_title("Sessions by Hour and Traffic Channel", weight="bold")
ax.set_xlabel("Channel")
ax.set_ylabel("Hour of day (24-hour clock)")
fig.tight_layout()
save_chart("traffic_by_hour_channel.png")
plt.show()

In [ ]:
peak_hours = (
    df.groupby(["channel group", "Hour"], as_index=False)["Sessions"]
    .sum()
    .sort_values(["channel group", "Sessions", "Hour"], ascending=[True, False, True])
    .groupby("channel group", as_index=False)
    .first()
    .sort_values("Sessions", ascending=False)
)
peak_hours["Peak Hour"] = peak_hours["Hour"].map(lambda hour: f"{hour:02d}:00")
display(peak_hours[["channel group", "Peak Hour", "Sessions"]])
print("These are peak clock hours aggregated across the observed date range.")

# Question 7 — Is there any correlation between high traffic (sessions) and high engagement rate over time?

In [ ]:
time_analysis = (
    df.groupby("DateHour", as_index=False)
    .agg(Sessions=("Sessions", "sum"), Engaged_Sessions=("Engaged Sessions", "sum"))
    .sort_values("DateHour")
)
time_analysis["Engagement rate"] = (
    time_analysis["Engaged_Sessions"] / time_analysis["Sessions"]
)
correlation = time_analysis["Sessions"].corr(time_analysis["Engagement rate"])
print(f"Hourly timestamp observations: {len(time_analysis):,}")
print(f"Pearson correlation (sessions vs engagement rate): {correlation:.3f}")
print("This descriptive association is not evidence that traffic volume causes engagement.")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sns.regplot(
    data=time_analysis,
    x="Sessions",
    y="Engagement rate",
    scatter_kws={"alpha": 0.55, "s": 28},
    line_kws={"color": "#d1495b", "linewidth": 2},
    ax=ax,
)
ax.set_title("Hourly Sessions vs Engagement Rate Over Time", weight="bold")
ax.set_xlabel("Sessions per timestamp (all channels)")
ax.set_ylabel("Weighted engagement rate")
fig.tight_layout()
save_chart("sessions_vs_engagement_rate.png")
plt.show()
print(f"Correlation interpretation: weak positive association (r = {correlation:.3f}).")

# Additional Business Analysis

The Syntecxhub project brief also mentions traffic sources, engagement, conversions, and page performance. The supplied dataset contains channel and engagement metrics, but it does **not** contain conversion/goal or page-level fields. Therefore, conversion rates, goal completions, and high/low-performing pages should not be invented from this dataset.


In [ ]:
business_summary = df.groupby("channel group").agg(
    Users=("Users", "sum"),
    Sessions=("Sessions", "sum"),
    Engaged_Sessions=("Engaged Sessions", "sum"),
    Non_Engaged_Sessions=("Non-Engaged Sessions", "sum"),
    Event_Count=("Event count", "sum"),
)
time_totals = df.assign(
    _weighted_time=df["Average engagement time per session"] * df["Sessions"]
).groupby("channel group").agg(
    weighted_seconds=("_weighted_time", "sum"),
    valid_time_sessions=("Sessions", lambda values: values[df.loc[values.index, "Average engagement time per session"].notna()].sum()),
)
business_summary["Weighted_Avg_Engagement_Time_Seconds"] = (
    time_totals["weighted_seconds"] / time_totals["valid_time_sessions"]
)
business_summary["Engagement_Rate"] = (
    business_summary["Engaged_Sessions"] / business_summary["Sessions"]
)
business_summary = business_summary.sort_values("Sessions", ascending=False)
display(business_summary)
print("Users are summed over channel-hour rows and are not a deduplicated reach metric.")

# Key Findings

Run the following cell after all calculations to generate concise, data-driven findings for the final report.


In [ ]:
top_traffic = channel_users.idxmax()
best_time_channel = avg_time_by_channel.idxmax()
best_engagement_channel = engagement_by_channel.idxmax()
weakest_engagement_channel = engagement_by_channel.idxmin()
peak_overall = trend.loc[trend["Sessions"].idxmax()]
peak_hour_by_channel = dict(zip(peak_hours["channel group"], peak_hours["Peak Hour"]))

print("DATA-DRIVEN FINDINGS")
print(f"Top traffic source by summed users: {top_traffic} ({channel_users.max():,.0f}).")
print(
    f"Highest engagement rate: {best_engagement_channel} "
    f"({engagement_by_channel.max():.1%}; {session_by_channel.loc[best_engagement_channel, 'Sessions']:,} sessions)."
)
print(
    f"Lowest engagement rate: {weakest_engagement_channel} "
    f"({engagement_by_channel.min():.1%}; {session_by_channel.loc[weakest_engagement_channel, 'Sessions']:,} sessions)."
)
print(
    f"Highest weighted engagement time: {best_time_channel} "
    f"({avg_time_by_channel.max():.1f} seconds; "
    f"{session_by_channel.loc[best_time_channel, 'Sessions']:,} sessions)."
)
print(
    f"Peak traffic timestamp: {peak_overall['DateHour']:%Y-%m-%d %H:%M} "
    f"({peak_overall['Sessions']:,} sessions)."
)
print("Peak clock hour by channel:", "; ".join(f"{channel}: {hour}" for channel, hour in peak_hour_by_channel.items()))
print(f"Sessions / engagement correlation: r = {correlation:.3f} (weak positive, non-causal).")

recommendations = [
    f"Maintain and experimentally optimize Organic Social, the largest traffic source ({channel_sessions['Organic Social']:,} sessions), rather than assuming volume guarantees quality.",
    f"Review Referral partners and landing-page journeys; Referral delivered {engagement_by_channel['Referral']:.1%} engagement across {channel_sessions['Referral']:,} sessions.",
    f"Test measured Organic Video growth: its {engagement_by_channel['Organic Video']:.1%} engagement rate and {avg_time_by_channel['Organic Video']:.0f}-second average are based on only {channel_sessions['Organic Video']:,} sessions.",
    f"Audit campaign tagging and audience/landing-page fit for Unassigned, which had {engagement_by_channel['Unassigned']:.1%} engagement across {channel_sessions['Unassigned']:,} sessions.",
    "Use the channel-specific peak hours as test windows for publishing and campaigns, and validate against additional weeks before reallocating spend.",
]
print("\nFIVE ACTIONABLE RECOMMENDATIONS")
for number, recommendation in enumerate(recommendations, start=1):
    print(f"{number}. {recommendation}")
print(
    "\nNot calculable from the supplied fields: bounce rate, conversion rate, goal completions, "
    "page performance, campaign ROI, or device-level performance."
)

## Conclusion, limitations, and dashboard outputs

The results above are calculated from the supplied channel-hour records. Summed `Users` values are not unique-user reach across time. Engagement-rate comparisons use engaged sessions divided by sessions; very small-volume channels should be interpreted cautiously. The correlation is descriptive and does not establish causation.

The CSV does not contain bounce rate, conversions, goal completions, campaign cost, landing pages, device, or page-performance fields, so these metrics cannot be calculated. The final cell exports dashboard-ready CSVs and a self-contained HTML report without requiring Plotly or Streamlit.

## Dashboard-ready exports

Run this final cell to write the offline HTML dashboard, clean summary CSVs, and chart copies.

In [ ]:
import base64
import html
import shutil

# Save filterable analysis data without changing the supplied source CSV.
df.to_csv(DASHBOARD_DIR / "traffic_channel_hour.csv", index=False)
business_summary.rename_axis("Channel").to_csv(DASHBOARD_DIR / "channel_summary.csv")
(
    df.groupby(["Hour", "channel group"], as_index=False)["Sessions"]
    .sum()
    .rename(columns={"channel group": "Channel"})
    .to_csv(DASHBOARD_DIR / "hourly_channel_traffic.csv", index=False)
)
time_analysis.to_csv(DASHBOARD_DIR / "sessions_engagement_trend.csv", index=False)
kpis.to_csv(DASHBOARD_DIR / "kpi_summary.csv", index=False)

chart_names = [
    "users_sessions_over_time.png",
    "users_by_channel.png",
    "sessions_by_channel.png",
    "avg_engagement_time_by_channel.png",
    "engagement_rate_by_channel.png",
    "engagement_rate_distribution_by_channel.png",
    "engaged_vs_non_engaged.png",
    "traffic_by_hour_channel.png",
    "sessions_vs_engagement_rate.png",
]
missing_charts = [name for name in chart_names if not (VISUALIZATION_DIR / name).is_file()]
if missing_charts:
    raise FileNotFoundError(f"Expected chart files were not generated: {missing_charts}")
for chart_name in chart_names:
    shutil.copy2(VISUALIZATION_DIR / chart_name, DASHBOARD_DIR / chart_name)

channel_options_html = "".join(
    f"<option value=\"{html.escape(channel, quote=True)}\">{html.escape(channel)}</option>"
    for channel in sorted(df["channel group"].unique())
)
rows_json = df[
    [
        "DateHour",
        "channel group",
        "Users",
        "Sessions",
        "Engaged Sessions",
        "Average engagement time per session",
    ]
].to_json(orient="records", date_format="iso")
card_specs = [
    ("Total Users (summed records)", "kpi-users"),
    ("Total Sessions", "kpi-sessions"),
    ("Total Engaged Sessions", "kpi-engaged"),
    ("Overall Engagement Rate", "kpi-rate"),
    ("Weighted Average Engagement Time", "kpi-time"),
    ("Traffic Channels", "kpi-channels"),
    ("Selected Date Range", "kpi-range"),
]
cards_html = "".join(
    f"<article class='card'><div class='label'>{html.escape(label)}</div>"
    f"<div class='value' id='{element_id}'>-</div></article>"
    for label, element_id in card_specs
)

dashboard_template = """<!doctype html>
<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">
<title>Website Traffic Analysis Dashboard</title>
<style>
body{font-family:Arial,sans-serif;margin:0;background:#f3f6fa;color:#1f2937}header{padding:28px 5%;background:#12304a;color:white}main{max-width:1240px;margin:24px auto;padding:0 20px}.cards{display:grid;grid-template-columns:repeat(auto-fit,minmax(180px,1fr));gap:14px}.card,.chart,.note,.filters,.table-wrap{background:white;border-radius:10px;padding:18px;box-shadow:0 2px 10px #17324d12}.label{font-size:13px;color:#607080}.value{font-size:24px;font-weight:700;margin-top:8px}.charts{display:grid;grid-template-columns:repeat(auto-fit,minmax(440px,1fr));gap:16px;margin-top:18px}.chart img{width:100%;height:auto}.chart h2{font-size:17px}.note{margin:18px 0;line-height:1.6}.filters{display:flex;gap:14px;align-items:end;flex-wrap:wrap;margin:18px 0}.filters label{display:grid;gap:5px;font-size:13px;color:#607080}.filters select,.filters input,.filters button{font:inherit;padding:8px;border:1px solid #c7d1dc;border-radius:5px;background:white}.filters button{cursor:pointer}.table-wrap{overflow-x:auto;margin-top:14px}table{width:100%;border-collapse:collapse}th,td{text-align:right;padding:9px;border-bottom:1px solid #e5eaf0}th:first-child,td:first-child{text-align:left}footer{padding:20px 5%;color:#607080;font-size:13px}
</style></head><body>
<header><h1>Website Traffic Analysis</h1><div>Syntecxhub Internship - supplied channel-hour dataset</div></header>
<main><h2>Filtered key performance indicators</h2><div class="filters">
<label>Channel<select id="channel-filter"><option value="all">All channels</option>__CHANNEL_OPTIONS__</select></label>
<label>Start date<input id="start-date" type="date" min="__MIN_DATE__" max="__MAX_DATE__" value="__MIN_DATE__"></label>
<label>End date<input id="end-date" type="date" min="__MIN_DATE__" max="__MAX_DATE__" value="__MAX_DATE__"></label>
<button id="reset-filter" type="button">Reset filters</button></div>
<div class="cards">__CARDS__</div>
<h2>Filtered channel summary</h2><div class="table-wrap"><table><thead><tr><th>Channel</th><th>Users (summed)</th><th>Sessions</th><th>Engaged</th><th>Non-engaged</th><th>Engagement rate</th><th>Avg engagement time (sec)</th></tr></thead><tbody id="channel-summary"></tbody></table></div>
<div class="note"><strong>Reading the data:</strong> Users are summed across channel-hour records and are not deduplicated unique people. The filters above update KPIs and the channel summary. The charts below summarize the full date range. Engagement rates are computed as total engaged sessions divided by sessions. Organic Video has a small session base; correlation is descriptive, not causal. CSV files beside this report can be filtered further in Excel or Power BI.</div>
<div class="charts">__FIGURES__</div></main>
<footer>Generated by the Website Traffic Analysis notebook. Filters run locally in your browser; no external services or dashboard packages are required.</footer>
<script>
const trafficRows = __ROWS__;
const channelFilter = document.getElementById("channel-filter");
const startFilter = document.getElementById("start-date");
const endFilter = document.getElementById("end-date");
const numberFormat = new Intl.NumberFormat();
function safeText(value) { return String(value).replace(/[&<>]/g, char => ({"&":"&amp;","<":"&lt;",">":"&gt;"}[char])); }
function refreshDashboard() {
  const channel = channelFilter.value;
  const start = startFilter.value;
  const end = endFilter.value;
  const selected = trafficRows.filter(row => {
    const date = row.DateHour.slice(0, 10);
    return (channel === "all" || row["channel group"] === channel) && (!start || date >= start) && (!end || date <= end);
  });
  const sum = field => selected.reduce((total, row) => total + Number(row[field] || 0), 0);
  const sessions = sum("Sessions");
  const engaged = sum("Engaged Sessions");
  const users = sum("Users");
  const weightedTime = selected.reduce((total, row) => total + Number(row["Average engagement time per session"] || 0) * Number(row.Sessions || 0), 0);
  document.getElementById("kpi-users").textContent = numberFormat.format(users);
  document.getElementById("kpi-sessions").textContent = numberFormat.format(sessions);
  document.getElementById("kpi-engaged").textContent = numberFormat.format(engaged);
  document.getElementById("kpi-rate").textContent = sessions ? (100 * engaged / sessions).toFixed(1) + "%" : "-";
  document.getElementById("kpi-time").textContent = sessions ? (weightedTime / sessions).toFixed(1) + " sec" : "-";
  document.getElementById("kpi-channels").textContent = numberFormat.format(new Set(selected.map(row => row["channel group"])).size);
  document.getElementById("kpi-range").textContent = selected.length ? start + " to " + end : "No matching data";
  const grouped = new Map();
  for (const row of selected) {
    const key = row["channel group"];
    if (!grouped.has(key)) grouped.set(key, {users:0, sessions:0, engaged:0, weightedTime:0});
    const item = grouped.get(key);
    item.users += Number(row.Users || 0);
    item.sessions += Number(row.Sessions || 0);
    item.engaged += Number(row["Engaged Sessions"] || 0);
    item.weightedTime += Number(row["Average engagement time per session"] || 0) * Number(row.Sessions || 0);
  }
  const body = document.getElementById("channel-summary");
  body.innerHTML = [...grouped.entries()].sort((a,b) => b[1].sessions-a[1].sessions).map(([name,item]) =>
    "<tr><td>" + safeText(name) + "</td><td>" + numberFormat.format(item.users) + "</td><td>" + numberFormat.format(item.sessions) + "</td><td>" + numberFormat.format(item.engaged) + "</td><td>" + numberFormat.format(item.sessions-item.engaged) + "</td><td>" + (item.sessions ? (100*item.engaged/item.sessions).toFixed(1)+"%" : "-") + "</td><td>" + (item.sessions ? (item.weightedTime/item.sessions).toFixed(1) : "-") + "</td></tr>"
  ).join("");
}
for (const element of [channelFilter, startFilter, endFilter]) element.addEventListener("change", refreshDashboard);
document.getElementById("reset-filter").addEventListener("click", () => {channelFilter.value="all"; startFilter.value="__MIN_DATE__"; endFilter.value="__MAX_DATE__"; refreshDashboard();});
refreshDashboard();
</script></body></html>"""

min_date = str(df["DateHour"].min().date())
max_date = str(df["DateHour"].max().date())
figures = []
for chart_name in chart_names:
    encoded = base64.b64encode((DASHBOARD_DIR / chart_name).read_bytes()).decode("ascii")
    figures.append(
        f"<section class='chart'><h2>{html.escape(chart_name.replace('_', ' ').removesuffix('.png').title())}</h2>"
        f"<img src='data:image/png;base64,{encoded}' alt='{html.escape(chart_name)}'></section>"
)

report_html = (
    dashboard_template
    .replace("__CHANNEL_OPTIONS__", channel_options_html)
    .replace("__CARDS__", cards_html)
    .replace("__FIGURES__", "".join(figures))
    .replace("__ROWS__", rows_json)
    .replace("__MIN_DATE__", min_date)
    .replace("__MAX_DATE__", max_date)
)
(DASHBOARD_DIR / "dashboard.html").write_text(report_html, encoding="utf-8")
print("Dashboard-ready files written to:", DASHBOARD_DIR)
print("Charts saved to:", VISUALIZATION_DIR)
print("Open dashboard/dashboard.html in a browser to view the filtered offline dashboard.")

In [ ]:
# Final project checks catch missing outputs before sharing the internship submission.
required_clean_columns = ["channel group", "DateHour", "Users", "Sessions", "Engaged Sessions"]
assert not df[required_clean_columns].isna().any().any()
assert (df["Engaged Sessions"] <= df["Sessions"]).all()
assert int(df["Sessions"].sum()) == int(kpis.loc[kpis["Metric"] == "Total Sessions", "Value"].iloc[0])

expected_files = [
    VISUALIZATION_DIR / name for name in chart_names
] + [
    DASHBOARD_DIR / "dashboard.html",
    DASHBOARD_DIR / "traffic_channel_hour.csv",
    DASHBOARD_DIR / "channel_summary.csv",
    DASHBOARD_DIR / "hourly_channel_traffic.csv",
    DASHBOARD_DIR / "sessions_engagement_trend.csv",
    DASHBOARD_DIR / "kpi_summary.csv",
]
missing_or_empty = [str(path) for path in expected_files if not path.is_file() or path.stat().st_size == 0]
assert not missing_or_empty, f"Missing or empty project outputs: {missing_or_empty}"

html_report = (DASHBOARD_DIR / "dashboard.html").read_text(encoding="utf-8")
assert 'id="channel-filter"' in html_report and 'id="start-date"' in html_report
assert "refreshDashboard()" in html_report
assert len(pd.read_csv(DASHBOARD_DIR / "traffic_channel_hour.csv")) == len(df)

print("FINAL QUALITY CHECK: PASS")
print(f"Clean rows: {len(df):,}; channels: {df['channel group'].nunique()}; charts: {len(chart_names)}")
print(f"Dashboard filters verified; checked {len(expected_files)} non-empty output files.")